## 9. Sensitivity analysis – does our choice depend on the assumptions?
The hospitalisation probabilities and vaccine effectiveness are estimates. We recompute the cost (and the personal thresholds) under alternative assumptions, including the **alternative literature estimates from one of our team members**. If the best rule stays the same, our decision is robust.

In [ ]:
TEAM_ALT_RISK = {"18": 0.005, "35": 0.005, "45": 0.008, "55": 0.010, "65": 0.025}   # team member's estimates
age2 = train["age_group"].astype(str).str[:2]
ve_alt = np.where(age2 == "65", 0.22, 0.35)                                          # lower VE for 65+

scenarios = {   # name: (h array, VE)
    "Base (CDC risk, VE 0.23, chronic x(1+n))": (hosp_prob(train), VE),
    "VE = 0.10 (very weak vaccine)":          (hosp_prob(train), 0.10),
    "VE = 0.20 (team-member estimate)":       (hosp_prob(train), 0.20),
    "VE = 0.40 (all-strain VE)":              (hosp_prob(train), 0.40),
    "65+ risk 5 % instead of 9 %":            (hosp_prob(train, {**AGE_RISK, "65": 0.05}), VE),
    "Weaker chronic effect x(1+0.5n)":        (hosp_prob(train, chronic_mult=0.5), VE),
    "Team-member estimates (65+ 2.5 %, VE 0.35/0.22, no chronic)": (hosp_prob(train, TEAM_ALT_RISK, 0.0), ve_alt),
}
top3 = model_table.index[:3].tolist()
rows = []
for sc, (h, ve) in scenarios.items():
    row = {"scenario": sc,
           "send to nobody": cost_per_person(y, np.zeros(n), h, ve),
           "send to everybody": cost_per_person(y, np.ones(n), h, ve)}
    for m in top3:
        row[f"{m} | personal"] = cost_per_person(y, OOF[m] > personal_threshold(h, ve), h, ve)
        row[f"{m} | 0.5"] = cost_per_person(y, OOF[m] > 0.5, h, ve)
    rows.append(row)
sens = pd.DataFrame(rows).set_index("scenario")
display(sens.round(1).T)
print("Best option in each scenario:")
print(sens.idxmin(axis=1).to_string())

**Interpretation.** Under every scenario the personal-threshold rule of a top model is the cheapest option, even though the absolute costs change with the assumptions. Under the team member's lower risk estimates, vaccines are worth sending to fewer people, so the saving over "send to nobody" is smaller – but our rule still adapts correctly because the thresholds are recomputed from the assumptions. This shows our approach is **robust to uncertainty in the medical estimates**.

## 10. Submission files – top 5 results
We take the **best decision rule for each model**, keep the **best version of each model family** (e.g. default vs tuned XGBoost), rank the families by cross-validated cost, and submit the **top 5 different model families**, so all five submissions are genuinely different approaches. Each model is refitted on **all** training data before predicting the test set. Files contain **0/1 decisions** in the format of the example submission.

In [ ]:
all_models = {**models, **tuned}
all_X = {name: X_eng for name in all_models}
for name, (kwargs, _) in experiments.items():                     # Logistic Regression variants
    all_models[name] = logreg(**kwargs); all_X[name] = experiments[name][1]
X_test_for = lambda name: X_test_eng if all_X[name] is X_eng else X_test_raw

def family(name):
    """Model family, so that tuned/untuned or pre-processing variants count as one approach."""
    for fam in ["LightGBM", "XGBoost", "HistGradientBoosting", "Random Forest", "Extra Trees", "Decision Tree", "LogReg"]:
        if name.startswith(fam):
            return fam
    return name

best_per_model = ranking.sort_values("mean").drop_duplicates("model").copy()
best_per_model["family"] = best_per_model["model"].map(family)
top5 = best_per_model.drop_duplicates("family").head(5).reset_index(drop=True)   # best version of each family
display(top5[["family", "model", "rule", "mean", "std"]].round(2))

submission_log = []
for order, row in enumerate(top5.itertuples(), start=1):
    model = clone(all_models[row.model]).fit(all_X[row.model], y)        # refit on the full training set
    p_test = model.predict_proba(X_test_for(row.model))[:, 1]
    if row.rule == "personal threshold":
        threshold = personal_threshold(h_test)
    elif row.rule == "global threshold":
        threshold = res.loc[res.model == row.model, "global_t"].median()   # typical cut-off chosen in CV
    else:
        threshold = 0.5
    decisions = (p_test > threshold).astype(int)
    sub = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET: decisions})
    fname = f"challenge_submission_group_{GROUP_NUMBER}_order_{order}.csv"
    sub.to_csv(fname, index=False)
    submission_log.append({"order": order, "file": fname, "model": row.model, "rule": row.rule,
                           "CV cost ($/person)": round(row.mean, 2), "CV std": round(row.std, 2),
                           "share sent a vaccine": f"{decisions.mean():.1%}"})
log_df = pd.DataFrame(submission_log)
display(log_df)

In [ ]:
# Safeguards: format, IDs, values, and that all five submissions are different
subs = [pd.read_csv(f) for f in log_df["file"]]
for f, s in zip(log_df["file"], subs):
    assert list(s.columns) == list(sample.columns), f"{f}: wrong columns"
    assert s[ID_COL].equals(sample[ID_COL]), f"{f}: IDs differ from the example submission"
    assert set(s[TARGET].unique()) <= {0, 1}, f"{f}: values must be 0/1"
for i in range(1, 5):
    print(f"order_{i+1} differs from order_1 for {(subs[i][TARGET] != subs[0][TARGET]).sum()} people")
print("All submission checks passed.")

In [ ]:
# Download the five files (Colab) – also commit them to the GitHub repository
try:
    from google.colab import files
    for f in log_df["file"]:
        files.download(f)
except ImportError:
    print("Not running in Colab – files are saved in the working directory.")